# MMAudio ile Text-to-Audio

**MMAudio** metin (ve isteğe bağlı video) girdisinden ses üreten bir yapay zeka modelidir.

Bu notebook'ta:
- Bir metin açıklaması yazacaksınız (örn. "yağmur sesi")
- MMAudio bunu gerçekçi bir ses dosyasına çevirecek
- Sonucu `.flac` olarak indirip dinleyebileceksiniz

**Model:** `large_44k_v2` (44kHz, yüksek kalite)  
**Lisans:** Model ağırlıkları CC-BY-NC 4.0 (yalnızca ticari olmayan kullanım)  
**GitHub:** https://github.com/hkchengrex/MMAudio

## GPU Ayarı

Bu notebook GPU gerektirir. T4 GPU (ücretsiz) yeterlidir.

1. Üst menüden **Runtime** → **Change runtime type** tıklayın
2. **Hardware accelerator** altından **T4 GPU** seçin
3. **Save** tıklayın

Aşağıdaki hücreyi çalıştırarak GPU'nun bağlı olduğunu doğrulayın.

In [ ]:
# GPU bagli mi kontrol et
import torch

if torch.cuda.is_available():
    # GPU adini ve bellek bilgisini yazdir
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_memory = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / (1024**3)
    print(f"GPU bulundu: {gpu_name}")
    print(f"GPU bellegi: {gpu_memory:.1f} GB")
else:
    # GPU yoksa uyari ver
    print("UYARI: GPU bulunamadi!")
    print("Lutfen Runtime > Change runtime type > T4 GPU secin.")
    print("GPU olmadan model cok yavas calisir.")

## Kurulum

Şimdi MMAudio reposunu klonlayıp gerekli kütüphaneleri yükleyeceğiz.  
Bu adım birkaç dakika sürebilir.

In [ ]:
# === TESHIS: Hangi paket CUDA torch'u bozuyor? ===
# Her paket kurulumundan sonra AYRI bir python prosesinde torch durumunu kontrol edecegiz.
# (reload torch calismadigi icin subprocess kullaniyoruz)

import subprocess

def cuda_kontrol(adim):
    """Yeni bir python prosesinde torch CUDA durumunu kontrol et"""
    sonuc = subprocess.run(
        ["python", "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.__file__)"],
        capture_output=True, text=True
    )
    satirlar = sonuc.stdout.strip().split("\n")
    if len(satirlar) == 3:
        versiyon, cuda, konum = satirlar
        durum = "AKTIF" if cuda == "True" else "BOZUK!"
        print(f"[{adim}] torch={versiyon} | CUDA={durum} | {konum}")
        if cuda != "True":
            print(f"  >>> SUCLU BULUNDU: '{adim}' adiminda CUDA kayboldu! <<<")
    else:
        print(f"[{adim}] HATA: {sonuc.stdout} {sonuc.stderr}")

# 0) Baslangic durumu
print("=" * 60)
cuda_kontrol("BASLANGIC")
print("=" * 60)

# 1) MMAudio klonla ve kur
!git clone https://github.com/hkchengrex/MMAudio.git
%cd MMAudio
!pip install -e . --no-deps -q
cuda_kontrol("pip install -e . --no-deps")

# 2) Her paketi tek tek kur
paketler = [
    "open_clip_torch",
    "transformers",
    "accelerate",
    "av",
    "colorlog",
    "hydra-core",
    "hydra-colorlog",
    "nitrous-ema",
    "tensordict",
    "torchdiffeq",
]

for paket in paketler:
    print(f"\n--- {paket} kuruluyor ---")
    !pip install {paket} -q
    cuda_kontrol(paket)

print("\n" + "=" * 60)
print("TESHIS TAMAMLANDI")
print("=" * 60)

In [ ]:
# Model agirliklarini indir (ilk seferde ~2-3 dakika surer)
print("Model indiriliyor, lutfen bekleyin...")

from mmaudio.eval_utils import all_model_cfg

# large_44k_v2 modelini sec
model_cfg = all_model_cfg['large_44k_v2']

# Model dosyalari yoksa otomatik indirir
model_cfg.download_if_needed()

print("Model basariyla indirildi!")

## Ses Üretimi Ayarları

Aşağıdaki hücrede iki değişken var:
- **PROMPT** — üretmek istediğiniz sesi tarif edin (İngilizce)
- **DURATION** — ses süresi (saniye cinsinden, max 8)

Bunları istediğiniz gibi değiştirip tekrar çalıştırabilirsiniz.

**Örnek prompt'lar:**
- `"rain falling on a rooftop at night"`
- `"a dog barking in a park"`
- `"ocean waves crashing on rocks"`
- `"thunder and lightning during a storm"`

In [ ]:
# ==============================
# BU DEGERLERI ISTEDIGINIZ GIBI DEGISTIRIN
# ==============================

# Uretmek istediginiz sesi tarif edin (Ingilizce)
PROMPT = "rain falling on a rooftop at night"

# Ses suresi (saniye) - maksimum 8 saniye
DURATION = 8

# ==============================

print(f"Prompt: {PROMPT}")
print(f"Sure: {DURATION} saniye")

In [ ]:
# Ses uretimini baslat
import torch
import torchaudio
from mmaudio.eval_utils import setup_eval_logging, generate
from mmaudio.model.networks import get_my_mmaudio
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.model.utils.features_utils import FeaturesUtils

# TF32 hizlandirmayi aktif et
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

setup_eval_logging()

# Cihaz ve veri tipini ayarla (T4 icin float16 kullaniyoruz)
device = 'cuda'
dtype = torch.float16  # T4 GPU bfloat16 desteklemez

# Modeli yukle
print("Model yukleniyor...")
net = get_my_mmaudio(model_cfg.model_name).to(device, dtype).eval()
net.load_weights(torch.load(model_cfg.model_path, map_location=device, weights_only=True))

# Rastgele sayi ureteci (tekrarlanabilir sonuclar icin)
rng = torch.Generator(device=device)
rng.manual_seed(42)

# Flow matching ayari (25 adim)
fm = FlowMatching(min_sigma=0, inference_mode='euler', num_steps=25)

# Ozellik cikarici ayarla
feature_utils = FeaturesUtils(
    tod_vae_ckpt=model_cfg.vae_path,
    synchformer_ckpt=model_cfg.synchformer_ckpt,
    enable_conditions=True,
    mode=model_cfg.mode,
    bigvgan_vocoder_ckpt=model_cfg.bigvgan_16k_path,
    need_vae_encoder=False
)
feature_utils = feature_utils.to(device, dtype).eval()

# Ses suresini ayarla
seq_cfg = model_cfg.seq_cfg
seq_cfg.duration = DURATION
net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

# Ses uret (video yok, sadece metin)
print(f"Ses uretiliyor: '{PROMPT}'...")
with torch.inference_mode():
    audios = generate(
        clip_video=None,        # Video yok
        sync_video=None,        # Video yok
        text=[PROMPT],          # Metin prompt
        negative_text=[''],     # Negatif prompt (bos)
        feature_utils=feature_utils,
        net=net,
        fm=fm,
        rng=rng,
        cfg_strength=4.5,       # Rehberlik gucu
    )

# Sonucu kaydet
audio = audios.float().cpu()[0]
output_file = 'result.flac'
torchaudio.save(output_file, audio, seq_cfg.sampling_rate)

print(f"\nSes basariyla uretildi ve '{output_file}' olarak kaydedildi!")
print(f"Ornekleme hizi: {seq_cfg.sampling_rate} Hz")
print(f"Sure: {DURATION} saniye")

In [ ]:
# Dosyayi bilgisayariniza indirin
from google.colab import files

print("result.flac indiriliyor...")

# Dosya var mi kontrol et
import os
if os.path.exists('result.flac'):
    # Dosyayi indir
    files.download('result.flac')
    print("Indirme baslatildi! Tarayicinizin indirilenler klasorune bakin.")
else:
    print("HATA: result.flac bulunamadi!")
    print("Lutfen onceki hucreleri sirayla calistirdiginizdan emin olun.")